<a href="https://colab.research.google.com/github/Vivekb638/movie-recommendation-pyspark/blob/main/notebooks/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Mount Google Drive



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Set the project paths

In [2]:
import os

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "movie-recommendation-pyspark"
)

RAW_DIR = os.path.join(
    PROJECT_DIR, "data/raw/ml-1m"
)

PROCESSED_DIR = os.path.join(
    PROJECT_DIR, "data/processed"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Project paths configured!")

Project paths configured!


In [3]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark==3.5.5

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.21.0.3+1-1~24.04.1_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.21.0.3+1-1~24.04.1) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.21.0.3+1-1~24.04.1_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.21.0.3+1-1~24.04.1) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.21.0.3+1-1~24.04.1) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.017s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't

In [4]:
import sys
import pyspark

print("Python version:", sys.version)
print("PySpark version:", pyspark.__version__)

!java -version

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PySpark version: 3.5.5
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [5]:
import os

os.environ["JAVA_HOME"] = (
    "/usr/lib/jvm/java-17-openjdk-amd64"
)

os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" +
    os.environ["PATH"]
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MovieRecommendation-DataPreprocessing")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.5


In [6]:
ratings = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "ratings")
)

movies = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "movies")
)

users = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "users")
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [7]:
print("Ratings:", ratings.count())
print("Movies:", movies.count())
print("Users:", users.count())

Ratings: 1000209
Movies: 3883
Users: 6040


Explore the ratings dataset

In [8]:
ratings.show(10, truncate=False)

ratings.describe("rating").show()

print("Number of unique users:", ratings.select("userId").distinct().count())

print("Number of unique movies:", ratings.select("movieId").distinct().count())

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1193   |5.0   |978300760|
|1     |661    |3.0   |978302109|
|1     |914    |3.0   |978301968|
|1     |3408   |4.0   |978300275|
|1     |2355   |5.0   |978824291|
|1     |1197   |3.0   |978302268|
|1     |1287   |5.0   |978302039|
|1     |2804   |5.0   |978300719|
|1     |594    |4.0   |978302268|
|1     |919    |4.0   |978301368|
+------+-------+------+---------+
only showing top 10 rows

+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|           1000209|
|   mean| 3.581564453029317|
| stddev|1.1171018453732653|
|    min|               1.0|
|    max|               5.0|
+-------+------------------+

Number of unique users: 6040
Number of unique movies: 3706


Check missing values

In [9]:
from pyspark.sql.functions import col, sum

print("Missing values in ratings:")

ratings.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in ratings.columns
]).show()

print("Missing values in movies:")

movies.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in movies.columns
]).show()

print("Missing values in users:")

users.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in users.columns
]).show()

Missing values in ratings:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     0|      0|     0|        0|
+------+-------+------+---------+

Missing values in movies:
+-------+-----+------+
|movieId|title|genres|
+-------+-----+------+
|      0|    0|     0|
+-------+-----+------+

Missing values in users:
+------+------+---+----------+-------+
|userId|gender|age|occupation|zipCode|
+------+------+---+----------+-------+
|     0|     0|  0|         0|      0|
+------+------+---+----------+-------+



Check duplicate records

In [10]:
print(
    "Duplicate user-movie rating pairs:",
    ratings.groupBy("userId", "movieId")
           .count()
           .filter(col("count") > 1)
           .count()
)

print(
    "Duplicate movie IDs:",
    movies.groupBy("movieId")
          .count()
          .filter(col("count") > 1)
          .count()
)

print(
    "Duplicate user IDs:",
    users.groupBy("userId")
         .count()
         .filter(col("count") > 1)
         .count()
)

Duplicate user-movie rating pairs: 0
Duplicate movie IDs: 0
Duplicate user IDs: 0


Clean the ratings dataset

In [11]:
from pyspark.sql.functions import col

ratings_clean = (
    ratings
    .dropna(subset=["userId", "movieId", "rating"])
    .filter(col("rating").between(1.0, 5.0))
    .dropDuplicates(["userId", "movieId"])
)

print("Original ratings:", ratings.count())
print("Clean ratings:", ratings_clean.count())

Original ratings: 1000209
Clean ratings: 1000209


Validate user IDs and movie IDs

In [12]:
valid_user_ids = users.select("userId").distinct()
valid_movie_ids = movies.select("movieId").distinct()

invalid_users = (
    ratings_clean
    .join(valid_user_ids, on="userId", how="left_anti")
)

invalid_movies = (
    ratings_clean
    .join(valid_movie_ids, on="movieId", how="left_anti")
)

print("Ratings with invalid users:", invalid_users.count())
print("Ratings with invalid movies:", invalid_movies.count())

Ratings with invalid users: 0
Ratings with invalid movies: 0


Clean movies and users

In [13]:
movies_clean = (
    movies
    .dropna(subset=["movieId", "title"])
    .dropDuplicates(["movieId"])
)

users_clean = (
    users
    .dropna(subset=["userId"])
    .dropDuplicates(["userId"])
)

print("Clean movies:", movies_clean.count())
print("Clean users:", users_clean.count())

Clean movies: 3883
Clean users: 6040


Join ratings with movies

In [14]:
ratings_movies = ratings_clean.join(
    movies_clean,
    on="movieId",
    how="inner"
)

ratings_movies.show(10, truncate=False)

print("Ratings and movies joined:", ratings_movies.count())

+-------+------+------+---------+----------------------------------------+-------------------------------+
|movieId|userId|rating|timestamp|title                                   |genres                         |
+-------+------+------+---------+----------------------------------------+-------------------------------+
|1197   |1     |3.0   |978302268|Princess Bride, The (1987)              |Action|Adventure|Comedy|Romance|
|594    |1     |4.0   |978302268|Snow White and the Seven Dwarfs (1937)  |Animation|Children's|Musical   |
|2398   |1     |4.0   |978302281|Miracle on 34th Street (1947)           |Drama                          |
|2918   |1     |4.0   |978302124|Ferris Bueller's Day Off (1986)         |Comedy                         |
|1022   |1     |5.0   |978300055|Cinderella (1950)                       |Animation|Children's|Musical   |
|1961   |1     |5.0   |978301590|Rain Man (1988)                         |Drama                          |
|2028   |1     |5.0   |978301619|Savi

Join user information

In [15]:
user_movie_data = ratings_movies.join(
    users_clean,
    on="userId",
    how="inner"
)

user_movie_data.show(10, truncate=False)

print("Final joined dataset:", user_movie_data.count())

+------+-------+------+---------+----------------------------------------+-------------------------------+------+---+----------+-------+
|userId|movieId|rating|timestamp|title                                   |genres                         |gender|age|occupation|zipCode|
+------+-------+------+---------+----------------------------------------+-------------------------------+------+---+----------+-------+
|1     |1197   |3.0   |978302268|Princess Bride, The (1987)              |Action|Adventure|Comedy|Romance|F     |1  |10        |48067  |
|1     |594    |4.0   |978302268|Snow White and the Seven Dwarfs (1937)  |Animation|Children's|Musical   |F     |1  |10        |48067  |
|1     |2398   |4.0   |978302281|Miracle on 34th Street (1947)           |Drama                          |F     |1  |10        |48067  |
|1     |2918   |4.0   |978302124|Ferris Bueller's Day Off (1986)         |Comedy                         |F     |1  |10        |48067  |
|1     |1022   |5.0   |978300055|Cinderel

Extract individual movie genres

In [16]:
from pyspark.sql.functions import explode, split

movie_genres = (
    movies_clean
    .select(
        "movieId",
        "title",
        explode(split(col("genres"), r"\|")).alias("genre")
    )
)

movie_genres.show(20, truncate=False)

print("Total movie-genre records:", movie_genres.count())

+-------+----------------------------------+----------+
|movieId|title                             |genre     |
+-------+----------------------------------+----------+
|1      |Toy Story (1995)                  |Animation |
|1      |Toy Story (1995)                  |Children's|
|1      |Toy Story (1995)                  |Comedy    |
|2      |Jumanji (1995)                    |Adventure |
|2      |Jumanji (1995)                    |Children's|
|2      |Jumanji (1995)                    |Fantasy   |
|3      |Grumpier Old Men (1995)           |Comedy    |
|3      |Grumpier Old Men (1995)           |Romance   |
|4      |Waiting to Exhale (1995)          |Comedy    |
|4      |Waiting to Exhale (1995)          |Drama     |
|5      |Father of the Bride Part II (1995)|Comedy    |
|6      |Heat (1995)                       |Action    |
|6      |Heat (1995)                       |Crime     |
|6      |Heat (1995)                       |Thriller  |
|7      |Sabrina (1995)                    |Come

Calculate user rating statistics

In [17]:
from pyspark.sql.functions import avg, count, min, max

user_statistics = (
    ratings_clean
    .groupBy("userId")
    .agg(
        count("rating").alias("total_ratings"),
        avg("rating").alias("average_rating"),
        min("rating").alias("minimum_rating"),
        max("rating").alias("maximum_rating")
    )
)

user_statistics.show(10)

+------+-------------+------------------+--------------+--------------+
|userId|total_ratings|    average_rating|minimum_rating|maximum_rating|
+------+-------------+------------------+--------------+--------------+
|    12|           23|3.8260869565217392|           1.0|           5.0|
|    13|          108| 3.388888888888889|           1.0|           5.0|
|    14|           25|              3.32|           1.0|           5.0|
|    18|          305|3.6491803278688524|           1.0|           5.0|
|    38|          100|              3.58|           1.0|           5.0|
|    46|           41| 4.219512195121951|           1.0|           5.0|
|    67|           64|          4.296875|           2.0|           5.0|
|    70|           54|3.7037037037037037|           1.0|           5.0|
|    93|          220|2.9727272727272727|           1.0|           5.0|
|   107|          121| 3.743801652892562|           2.0|           5.0|
+------+-------------+------------------+--------------+--------

Calculate movie statistics

In [18]:
movie_statistics = (
    ratings_clean
    .groupBy("movieId")
    .agg(
        count("rating").alias("rating_count"),
        avg("rating").alias("average_rating")
    )
    .join(
        movies_clean.select("movieId", "title"),
        on="movieId",
        how="inner"
    )
)

movie_statistics.orderBy(
    col("rating_count").desc()
).show(20, truncate=False)

+-------+------------+------------------+-----------------------------------------------------+
|movieId|rating_count|average_rating    |title                                                |
+-------+------------+------------------+-----------------------------------------------------+
|2858   |3428        |4.3173862310385065|American Beauty (1999)                               |
|260    |2991        |4.453694416583082 |Star Wars: Episode IV - A New Hope (1977)            |
|1196   |2990        |4.292976588628763 |Star Wars: Episode V - The Empire Strikes Back (1980)|
|1210   |2883        |4.022892819979188 |Star Wars: Episode VI - Return of the Jedi (1983)    |
|480    |2672        |3.7638473053892216|Jurassic Park (1993)                                 |
|2028   |2653        |4.337353938937053 |Saving Private Ryan (1998)                           |
|589    |2649        |4.058512646281616 |Terminator 2: Judgment Day (1991)                    |
|2571   |2590        |4.315830115830116 

Calculate genre statistics

In [19]:
genre_statistics = (
    ratings_movies
    .select(
        "movieId",
        "rating",
        explode(split(col("genres"), r"\|")).alias("genre")
    )
    .groupBy("genre")
    .agg(
        count("rating").alias("total_ratings"),
        avg("rating").alias("average_rating")
    )
    .orderBy(col("total_ratings").desc())
)

genre_statistics.show(20, truncate=False)

+-----------+-------------+------------------+
|genre      |total_ratings|average_rating    |
+-----------+-------------+------------------+
|Comedy     |356580       |3.522098827752538 |
|Drama      |354529       |3.766332232342065 |
|Action     |257457       |3.4911849357368414|
|Thriller   |189680       |3.5704660480809784|
|Sci-Fi     |157294       |3.466521291339784 |
|Romance    |147523       |3.607464598740535 |
|Adventure  |133953       |3.477256948332624 |
|Crime      |79541        |3.708678543141273 |
|Horror     |76386        |3.215013222318226 |
|Children's |72186        |3.422034743579087 |
|War        |68527        |3.893326717935996 |
|Animation  |43293        |3.684868223500335 |
|Musical    |41533        |3.6655189849035708|
|Mystery    |40178        |3.6681019463387923|
|Fantasy    |36301        |3.447370595851354 |
|Western    |20683        |3.6377701493980563|
|Film-Noir  |18261        |4.075187558184108 |
|Documentary|7910         |3.933122629582807 |
+-----------+

Save the cleaned datasets to Google Drive

In [20]:
output_datasets = {
    "ratings_clean": ratings_clean,
    "movies_clean": movies_clean,
    "users_clean": users_clean,
    "user_movie_data": user_movie_data,
    "movie_genres": movie_genres,
    "user_statistics": user_statistics,
    "movie_statistics": movie_statistics,
    "genre_statistics": genre_statistics
}

for name, dataframe in output_datasets.items():
    output_path = os.path.join(PROCESSED_DIR, name)

    dataframe.write.mode("overwrite").parquet(output_path)

    print(f"Saved: {name}")

Saved: ratings_clean
Saved: movies_clean
Saved: users_clean
Saved: user_movie_data
Saved: movie_genres
Saved: user_statistics
Saved: movie_statistics
Saved: genre_statistics


Verify saved files

In [21]:
for name in output_datasets.keys():
    path = os.path.join(PROCESSED_DIR, name)

    df = spark.read.parquet(path)

    print(name, ":", df.count(), "records")

ratings_clean : 1000209 records
movies_clean : 3883 records
users_clean : 6040 records
user_movie_data : 1000209 records
movie_genres : 6408 records
user_statistics : 6040 records
movie_statistics : 3706 records
genre_statistics : 18 records
